In [1]:
%pip install pandas -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import pandas as pd

INPUT_CSV = "../data/processed/ner_candidate_sentences.csv"      # from the parser notebook
OUTPUT_CSV = "../data/processed/prioritized_sentences.csv"
TOP_N_FOR_ANNOTATION = 700                      # how many to actually annotate
TOP_N_OUTPUT_CSV = "../data/processed/annotation_batch_top700.csv"

df = pd.read_csv(INPUT_CSV)
print(df.shape)
df.head()

(9426, 4)


,file,specialty,section,sentence
0,0.html,Allergy / Immunology,MEDICATIONS,Her only medication currently is Ortho Tri-Cyc...
1,0.html,Allergy / Immunology,PLAN,She will try Zyrtec instead of Allegra again.
2,0.html,Allergy / Immunology,PLAN,Another option will be to use loratadine.
3,0.html,Allergy / Immunology,PLAN,She does not think she has prescription covera...
4,0.html,Allergy / Immunology,PLAN,Samples of Nasonex two sprays in each nostril ...


In [3]:
# Dosage: "500mg", "5 mg", "two sprays", "1 tablet", "10 ml"
DOSAGE_RE = re.compile(
    r"\b\d+(\.\d+)?\s?(mg|mcg|ml|g|units?|tablets?|tabs?|sprays?|drops?|puffs?|capsules?)\b",
    re.IGNORECASE,
)

# Frequency: "twice daily", "once a day", "every 6 hours", "1-0-1", "bid", "tid", "qid"
FREQUENCY_RE = re.compile(
    r"\b(once|twice|three times|four times)\s+(a\s+)?(day|daily)\b"
    r"|\bevery\s+\d+\s+hours?\b"
    r"|\b(bid|tid|qid|qd|prn)\b"
    r"|\b\d-\d-\d\b",
    re.IGNORECASE,
)

# Duration: "for 5 days", "for two weeks", "x3d", "for a month"
DURATION_RE = re.compile(
    r"\bfor\s+\d+\s+(day|days|week|weeks|month|months)\b"
    r"|\bfor\s+(a|one|two|three|four|five|six)\s+(day|days|week|weeks|month|months)\b"
    r"|\bx\d+d\b",
    re.IGNORECASE,
)

# Drug-like capitalized tokens NOT at sentence start (crude brand-name detector).
# Filters out common non-drug capitalized words often seen in clinical notes.
COMMON_NON_DRUG_CAPS = {
    "I", "She", "He", "The", "This", "They", "We", "It", "His", "Her",
    "MEDICATIONS", "PLAN", "TREATMENT", "RECOMMENDATIONS", "ASSESSMENT",
}

def count_dosage(s):
    return len(DOSAGE_RE.findall(s))

def count_frequency(s):
    return len(FREQUENCY_RE.findall(s))

def count_duration(s):
    return len(DURATION_RE.findall(s))

def count_druglike_tokens(s):
    tokens = s.split()
    count = 0
    for i, tok in enumerate(tokens):
        clean = tok.strip(",.;:()")
        if (
            i > 0                                  # skip sentence-initial word
            and clean.istitle()
            and clean.isalpha()
            and len(clean) > 2
            and clean not in COMMON_NON_DRUG_CAPS
        ):
            count += 1
    return count

In [4]:
df["dosage_signals"] = df["sentence"].apply(count_dosage)
df["frequency_signals"] = df["sentence"].apply(count_frequency)
df["duration_signals"] = df["sentence"].apply(count_duration)
df["druglike_tokens"] = df["sentence"].apply(count_druglike_tokens)

# Weighted score: dosage/frequency/duration are the highest-value signals
# since they're the hardest entities to synthesize well; druglike tokens
# get a lower weight since the heuristic is noisier (more false positives).
df["priority_score"] = (
    df["dosage_signals"] * 3
    + df["frequency_signals"] * 3
    + df["duration_signals"] * 2
    + df["druglike_tokens"] * 1
)

df = df.sort_values("priority_score", ascending=False).reset_index(drop=True)
df.head(15)

,file,specialty,section,sentence,dosage_signals,frequency_signals,duration_signals,druglike_tokens,priority_score
0,1960.html,Nephrology,MEDICATIONS,"Triglide 50 mg daily, Prevacid 30 mg daily, Pl...",17,0,0,11,62
1,526.html,Consult - History and Phy.,MEDICATIONS,From the list by the nursing home are Aricept ...,10,7,0,7,58
2,2069.html,Neurology,MEDICATIONS,From the list by the nursing home are Aricept ...,10,7,0,7,58
3,2023.html,Nephrology,MEDICATIONS,"Singulair 10 mg once daily, Cardizem 365 mg on...",9,7,0,6,54
4,748.html,Consult - History and Phy.,MEDICATIONS,"Singulair 10 mg once daily, Cardizem 365 mg on...",9,7,0,6,54
5,3217.html,Psychiatry / Psychology,DISCHARGE MEDICATIONS,"p.r.n., Lasix 80 mg daily, Isordil 10 mg t.i.d...",11,0,0,14,47
6,2137.html,Neurology,DISCHARGE MEDICATIONS,"p.r.n., Lasix 80 mg daily, Isordil 10 mg t.i.d...",11,0,0,14,47
7,1091.html,Discharge Summary,DISCHARGE MEDICATIONS,"p.r.n., Lasix 80 mg daily, Isordil 10 mg t.i.d...",11,0,0,14,47
8,1969.html,Nephrology,MEDICATIONS,"Starlix 120 mg b.i.d., Compazine b.i.d., aspir...",13,0,0,8,47
9,628.html,Consult - History and Phy.,CURRENT MEDICATIONS,"Acetaminophen 325 mg 2 tablets twice daily, Co...",9,4,0,6,45


In [5]:
def normalize(s):
    return re.sub(r"\s+", " ", s.lower().strip())

before = len(df)
df["_norm"] = df["sentence"].apply(normalize)
df = df.drop_duplicates(subset="_norm").drop(columns="_norm").reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} near-duplicate sentences ({before} -> {after})")

Removed 5000 near-duplicate sentences (9426 -> 4426)


In [6]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved full ranked list -> {OUTPUT_CSV} ({len(df)} rows)")

top_n = df.head(TOP_N_FOR_ANNOTATION)
top_n.to_csv(TOP_N_OUTPUT_CSV, index=False)
print(f"Saved top {len(top_n)} rows for annotation -> {TOP_N_OUTPUT_CSV}")

# Quick sanity check: how many of the top N actually have zero signals?
# (should be 0 or very close to it if your candidate pool is large enough)
zero_signal = (top_n["priority_score"] == 0).sum()
print(f"Rows in top {TOP_N_FOR_ANNOTATION} with priority_score == 0: {zero_signal}")

Saved full ranked list -> ../data/processed/prioritized_sentences.csv (4426 rows)
Saved top 700 rows for annotation -> ../data/processed/annotation_batch_top700.csv
Rows in top 700 with priority_score == 0: 0


In [7]:
df["priority_score"].describe()

count    4426.000000
mean        1.446227
std         3.719416
min         0.000000
25%         0.000000
50%         0.000000
75%         2.000000
max        62.000000
Name: priority_score, dtype: float64

In [8]:
%pip install matplotlib

  Using cached matplotlib-3.11.0-cp315-cp315-win_amd64.whl
  Using cached pillow-12.3.0-cp315-cp315-win_amd64.whl.metadata (9.3 kB)
Using cached pillow-12.3.0-cp315-cp315-win_amd64.whl (7.2 MB)

   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   ---------------------------------------- 0/2 [pillow]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- ------------------- 1/2 [matplotlib]
   -------------------- --------